# Cross-Domain Join Scenarios for Data Modeling

This notebook identifies **7 high-value table combination scenarios** across the Databricks catalog landscape spanning **5 domains**: Inventory, Sales, Supply Chain, Finance, and Master Data.

**Purpose**: Help discover joinable tables with shared columns to build new use cases — merging input and output variables to derive insights such as Expected Delivery Date, Fulfillment Risk, Forecast Accuracy, and more.

## Key Join Columns Across Domains
| Join Key | Bridges Domains | Table Count |
|---|---|---|
| `Item Number` | Inventory ↔ Sales ↔ Supply Chain ↔ Finance ↔ Master | 148 |
| `Customer Number` | Sales ↔ Finance ↔ Master ↔ Inventory | 51 |
| `Location` / `Warehouse` | Inventory ↔ Sales ↔ Supply Chain | 40+ |
| `Source` (GP/SAP) | All domains | 179 |
| `Product Class` / `Brand` | Inventory ↔ Master ↔ Sales ↔ Supply Chain | 44 |
| Date columns | All domains via `date_table` | 34 |

## Target Column Summary (Output Variables)
| Scenario | Target Column (Output) | Type | Description |
|---|---|---|---|
| 1. Fulfillment Risk | `fulfillment_risk_level` | Categorical | Classifies each backorder as Low / Medium / High risk based on supply visibility |
| 2. Root Cause | `margin_at_risk` | Numeric (Currency) | Gross margin value exposed due to backorders, segmented by customer & product |
| 3. Inventory Health | `stockout_risk_level` | Categorical | Classifies items as Critical / High / Medium / Low / Healthy based on stock + forecast |
| 4. Supply Projection | `supply_adequacy` | Categorical | Flags items as Insufficient Supply / Adequate but Tight / Surplus |
| 5. Lead Time Impact | `delay_severity` | Categorical | Classifies processing delays as On Track / Minor / Significant / Critical |
| 6. Pipeline Risk | `pipeline_fulfillment_risk` | Categorical | Flags open sales opportunities at risk due to inventory shortages |
| 7. Planning Trends | `stock_health_status` | Categorical | Identifies Chronic Understock / Chronic Overstock / Normal patterns over time |

## Scenario 1: Expected Delivery Date & Fulfillment Risk

**Tables**:
- `dev_inventory.gold.mv_usgp_item_inventory_enriched` (92 cols) — inventory position, lead times, expected dates
- `dev_sales.gold.vw_gpcurrentbackorder_enriched` (62 cols) — current open backorders with order dates

**Join Key**: `Item Number` (inventory) = `ItemNmbr` (backorders)

> **🎯 TARGET COLUMN**: `fulfillment_risk_level` — Categorical classification (Low / Medium / High) indicating whether an open backorder can be fulfilled based on current inventory, in-transit supply, and expected receipts.

**Supporting Derived Variables**:
- **Fulfillment Risk Score**: `DATEDIFF(Next Expected Receipt, ReqShipDate)` — negative = at risk
- **Estimated Delivery Date**: `OrdrDate + Purchasing Lead Time`
- **Supply Gap**: `BackorderQty - Available`
- **Days Overdue**: `DATEDIFF(CURRENT_DATE, ReqShipDate)`

In [0]:
%sql
-- SCENARIO 1: Expected Delivery Date & Fulfillment Risk
-- Joins inventory position with open backorders to calculate delivery risk

SELECT 
  bo.SOPNumbe                                       AS order_number,
  bo.ItemNmbr                                        AS item_number,
  bo.ItemDesc                                        AS item_description,
  bo.CustNmbr                                        AS customer_number,
  bo.CustName                                        AS customer_name,
  bo.SOPWarehouse                                    AS warehouse,
  bo.OrdrDate                                        AS order_date,
  bo.ReqShipDate                                     AS requested_ship_date,
  bo.BackorderQty                                    AS backorder_qty,
  bo.XTNDPRCE                                        AS extended_price,

  -- Inventory position
  inv.`On Hand`                                      AS on_hand_qty,
  inv.`Available`                                    AS available_qty,
  inv.`Back ordered`                                 AS total_backordered,
  inv.`On order`                                     AS on_order_qty,
  inv.`In Transit`                                   AS in_transit_qty,
  inv.`Safety`                                       AS safety_stock,

  -- Lead time & expected dates
  inv.`Purchasing Lead Time`                         AS purchasing_lead_time_days,
  inv.`Mfg Fixed Lead Time`                          AS mfg_fixed_lead_time,
  inv.`Next Expected Receipt`                        AS next_expected_receipt,
  inv.`Next Expected Delivery`                       AS next_expected_delivery,

  -- === DERIVED VARIABLES ===
  
  -- Estimated delivery date based on lead time
  DATE_ADD(CURRENT_DATE(), CAST(inv.`Purchasing Lead Time` AS INT)) 
                                                     AS estimated_delivery_date,

  -- Fulfillment risk: negative = ship date already passed
  DATEDIFF(inv.`Next Expected Receipt`, 
           TRY_CAST(bo.ReqShipDate AS DATE))         AS days_until_fulfillment,

  -- Supply gap: how much more is needed vs available
  bo.BackorderQty - inv.`Available`                  AS supply_gap_qty,

  -- Days overdue from requested ship date
  DATEDIFF(CURRENT_DATE(), 
           TRY_CAST(bo.ReqShipDate AS DATE))         AS days_overdue,

  -- Risk classification
  CASE 
    WHEN inv.`Available` >= bo.BackorderQty THEN 'Low - Stock Available'
    WHEN inv.`On order` > 0 OR inv.`In Transit` > 0 THEN 'Medium - Supply In Pipeline'
    WHEN inv.`Next Expected Receipt` IS NOT NULL 
     AND inv.`Next Expected Receipt` <= DATE_ADD(CURRENT_DATE(), 14) THEN 'Medium - Receipt Expected'
    ELSE 'High - No Supply Visibility'
  END                                                AS fulfillment_risk_level

FROM dev_sales.gold.vw_gpcurrentbackorder_enriched bo
LEFT JOIN dev_inventory.gold.mv_usgp_item_inventory_enriched inv
  ON bo.ItemNmbr = inv.`Item Number`
  AND bo.SOPWarehouse = inv.Location
WHERE bo.BackorderQty > 0
ORDER BY days_overdue DESC NULLS LAST
LIMIT 10000

order_number,item_number,item_description,customer_number,customer_name,warehouse,order_date,requested_ship_date,backorder_qty,extended_price,on_hand_qty,available_qty,total_backordered,on_order_qty,in_transit_qty,safety_stock,purchasing_lead_time_days,mfg_fixed_lead_time,next_expected_receipt,next_expected_delivery,estimated_delivery_date,days_until_fulfillment,supply_gap_qty,days_overdue,fulfillment_risk_level
2852492,97012,"BALLARD 24HR OC KIT CHG,Q2,7",103183,BEACON HEALTH SYSTEM,LVL FG,09/18/2026,09/23/2026,1.00000,131.58000,0.00000,0.00000,66.00000,0.00000,0.00000,234.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,1.00000,null,Medium - Receipt Expected
809001.5,5-5338-11,"LARYNGOSCOPE, HANDLE, FO, DISPOSABLE, STUBBY, STERILE",130,CARDINAL HEALTH,LVL FG,09/11/2026,09/28/2026,4.00000,840.00000,0.00000,0.00000,20.00000,0.00000,0.00000,0.00000,182,0,1900-01-01,1900-01-01,2027-04-14,null,4.00000,null,Medium - Receipt Expected
811954.3,4950F-7-0-25,"DIVIDED ADULT CAPNOGRAPY CANNULA, FEMALE LUER STYLE. 7' O2 & 2"" PIGTAIL CO2",HEPTNY,HENRY SCHEIN INC,PHX FG,09/21/2026,10/06/2026,7.00000,267.82000,324.00000,97.00000,0.00000,0.00000,0.00000,122.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,-90.00000,null,Low - Stock Available
784608.4,32640,"NASAL CANNULA, ADULT CONTOUR FIT, W/25' TUBING, MCKESSON",OCY001,CYPRESS MEDICAL PRODUCTS,PHX FG,06/26/2026,09/23/2026,9.00000,222.84000,315.00000,95.00000,2052.00000,0.00000,0.00000,0.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,-86.00000,null,Low - Stock Available
811319.2,A4U11XXX,"ANES CIRCUIT, ADULT, 108 IN EXP, 3L BAG",130,CARDINAL HEALTH,LVL FG,09/18/2026,10/05/2026,1.00000,160.05000,0.00000,0.00000,35.00000,0.00000,0.00000,18.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,1.00000,null,Medium - Receipt Expected
784609,32637,"NASAL CANNULA ADULT, CONTOUR FIT, W/7' TUBING, MCKESSON",OCY001,CYPRESS MEDICAL PRODUCTS,PHX FG,06/26/2026,10/09/2026,864.00000,15707.52000,719.00000,0.00000,18881.00000,0.00000,0.00000,0.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,864.00000,null,Medium - Receipt Expected
808470.6,7900-0-25,BUBBLE HUMIDIFIER UP TO 15 LPM W/ 6 PSI SAFETY VALVE. 25/CS,482,MEDLINE INDUSTRIES INC,ELP FG,09/10/2026,09/22/2026,5.00000,440.25000,8.00000,-1.00000,686.00000,0.00000,0.00000,702.00000,0,24,1900-01-01,1900-01-01,2026-10-14,null,6.00000,null,Medium - Receipt Expected
805215,ABC1004,"120"" ADULT DUAL LIMB EXPANDABLE CIRCUIT- OWENS & MINOR",725,OM MAIN STREET,PHX FG,08/31/2026,09/03/2026,1.00000,142.60000,0.00000,0.00000,52.00000,0.00000,0.00000,50.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,1.00000,null,Medium - Receipt Expected
2836244,NA11519,"108"" PEDIATRIC DUAL LIMB EXPANDABLE CIRCUIT- DRAEGER",C103142,DRAEGER MEDICAL,PHX FG,07/27/2026,10/09/2026,50.00000,6400.00000,57.00000,57.00000,50.00000,0.00000,0.00000,12.00000,0,28,1900-01-01,1900-01-01,2026-10-14,null,-7.00000,null,Low - Stock Available
811130.4,1-5075-32,LATEX FREE NASO A/W STER 32FR,700,OWENS & MINOR,LVL FG,09/18/2026,10/05/2026,2.00000,92.78000,0.00000,0.00000,24.00000,0.00000,0.00000,0.00000,154,0,1900-01-01,1900-01-01,2027-03-17,null,2.00000,null,Medium - Receipt Expected


## Scenario 2: Backorder Root Cause by Customer & Product

**Tables**:
- `dev_supply_chain.gold.vw_gpbackorderhistory` (11 cols) — historical backorder snapshots
- `dev_master.gold.vw_global_reporting_customer_master_enriched` (22 cols) — customer segmentation
- `dev_master.silver.vw_us_gp_item_master_enriched` (68 cols) — item attributes, brand, category

**Join Keys**: 
- `Customer Number` → customer master
- `Item Number` → item master

> **🎯 TARGET COLUMN**: `margin_at_risk` — Numeric (currency) value representing `SUM(Total Price) - SUM(Total Cost)` i.e. the gross margin exposed by backorders, segmented by customer channel, market, brand, and product category.

**Supporting Derived Variables**:
- `total_backorder_value` — Total revenue at risk
- `total_backorder_qty` — Volume of backordered units
- `distinct_customers` / `distinct_items` — Concentration metrics
- `avg_bo_qty_per_order` — Severity indicator per order line

In [0]:
%sql
-- SCENARIO 2: Backorder Root Cause Analysis by Customer & Product Segmentation
-- Joins backorder history with customer and item master for root cause dimensions

SELECT 
  -- Customer segmentation
  cust.`Customer Channel`,
  cust.`Customer Market`,
  cust.`Customer Type`,
  cust.`Facility Type`,
  cust.`Country`,
  cust.`international_region`,
  cust.`international_sub_region`,

  -- Product segmentation
  item.`Brand`,
  item.`Product Class`,
  item.`MktCategory`,
  item.`MktSubCategory`,
  item.`MktGroupMajor`,
  item.`Segment`,
  item.`Make/Buy`                                    AS make_or_buy,
  item.`Country of Origin`,

  -- Backorder metrics
  boh.`Product Allocated`,
  COUNT(DISTINCT boh.`Customer Number`)              AS distinct_customers,
  COUNT(DISTINCT boh.`Item Number`)                  AS distinct_items,
  SUM(boh.`Total BO Qty`)                            AS total_backorder_qty,
  SUM(boh.`Total Price`)                             AS total_backorder_value,
  SUM(boh.`Total Cost`)                              AS total_backorder_cost,
  SUM(boh.`Total Price`) - SUM(boh.`Total Cost`)     AS margin_at_risk,

  -- Root cause indicators
  AVG(boh.`Total BO Qty`)                            AS avg_bo_qty_per_order,
  COUNT(*)                                           AS backorder_line_count

FROM dev_supply_chain.gold.vw_gpbackorderhistory boh
LEFT JOIN dev_master.gold.vw_global_reporting_customer_master_enriched cust
  ON boh.`Customer Number` = cust.`Customer Number`
LEFT JOIN dev_master.silver.vw_us_gp_item_master_enriched item
  ON boh.`Item Number` = item.`Item Number`
GROUP BY ALL
ORDER BY total_backorder_value DESC
LIMIT 100

## Scenario 3: Inventory Health vs. Forecast Accuracy

**Tables**:
- `dev_inventory.gold.mv_usgp_item_inventory_enriched` (92 cols) — current inventory position, safety stock, order points
- `dev_supply_chain.gold.current_cons_forecast` (12 cols) — consensus demand forecast
- `dev_supply_chain.gold.forecast_variance` (14 cols) — forecast cycle-over-cycle changes

**Join Keys**: `Item Number` (inventory) ↔ `SIOP_Item` (forecast) — may require an item alias mapping

> **🎯 TARGET COLUMN**: `stockout_risk_level` — Categorical classification (Critical / High / Medium / Low / Healthy) based on the intersection of inventory position relative to safety stock AND forecast volatility.

**Supporting Derived Variables**:
- `stock_coverage_ratio` — `Available / Forecast` (how many periods current stock can cover)
- `below_safety_flag` — Boolean: `Available < Safety`
- `overstocked_flag` — Boolean: `On Hand > Max Inventory`

In [0]:
%sql
-- SCENARIO 3: Inventory Health vs Forecast Accuracy
-- Compares current inventory position against consensus forecast and variance

WITH latest_forecast AS (
  SELECT 
    SIOP_Item,
    ProductClass,
    Channel,
    SubRegion,
    Region_DC_Network,
    SUM(FC_Value)                                    AS total_forecast_value
  FROM dev_supply_chain.gold.current_cons_forecast
  GROUP BY SIOP_Item, ProductClass, Channel, SubRegion, Region_DC_Network
),
forecast_var AS (
  SELECT 
    ItemId0                                          AS item_id,
    Curr_cycle,
    prev_cycle,
    value_change,
    pct_change
  FROM dev_supply_chain.gold.forecast_variance
)

SELECT 
  inv.`Item Number`,
  inv.`Item Description`,
  inv.Location,
  inv.`Source`,

  -- Current inventory position
  inv.`On Hand`,
  inv.`Available`,
  inv.`Safety`                                       AS safety_stock,
  inv.`Order Point Qty`,
  inv.`On order`,
  inv.`In Transit`,
  inv.`Back ordered`,
  inv.`Replenishment Method`,
  inv.`Standard Cost`,
  inv.`Inventory Value`,

  -- Forecast data
  fc.total_forecast_value,
  fc.ProductClass                                    AS forecast_product_class,
  fc.Channel                                         AS forecast_channel,

  -- Forecast variance
  fv.value_change                                    AS forecast_value_change,
  fv.pct_change                                      AS forecast_pct_change,

  -- === DERIVED VARIABLES ===

  -- Stock coverage ratio (how many forecast periods can current stock cover)
  CASE 
    WHEN fc.total_forecast_value > 0 
    THEN ROUND(inv.`Available` / fc.total_forecast_value, 2)
    ELSE NULL 
  END                                                AS stock_coverage_ratio,

  -- Below safety stock flag
  CASE 
    WHEN inv.`Available` < inv.`Safety` THEN TRUE 
    ELSE FALSE 
  END                                                AS below_safety_flag,

  -- Overstocked flag
  CASE 
    WHEN inv.`On Hand` > inv.`Max Inventory` AND inv.`Max Inventory` > 0 THEN TRUE 
    ELSE FALSE 
  END                                                AS overstocked_flag,

  -- Stockout risk classification
  CASE 
    WHEN inv.`Available` <= 0 THEN 'Critical - Zero/Negative Availability'
    WHEN inv.`Available` < inv.`Safety` AND ABS(COALESCE(fv.pct_change, 0)) > 20 
      THEN 'High - Below Safety + Volatile Forecast'
    WHEN inv.`Available` < inv.`Safety` THEN 'Medium - Below Safety Stock'
    WHEN inv.`Available` < inv.`Order Point Qty` THEN 'Low - Below Reorder Point'
    ELSE 'Healthy'
  END                                                AS stockout_risk_level

FROM dev_inventory.gold.mv_usgp_item_inventory_enriched inv
LEFT JOIN latest_forecast fc
  ON inv.`Item Number` = fc.SIOP_Item
LEFT JOIN forecast_var fv
  ON inv.`Item Number` = fv.item_id
WHERE inv.`Inactive` = 0
ORDER BY stockout_risk_level, inv.`Available`
LIMIT 100

## Scenario 4: Global Projection & Supply Review (Jothishwar's New Tables)

**Tables** (created Oct 13, 2026):
- `prod_supply_chain.gold.global_projection_tool_detail` (42 cols) — weekly item-level projections with inventory breakdown
- `prod_supply_chain.gold.global_projection_tool` (9 cols) — summary projections
- `prod_supply_chain.gold.global_supply_review` (17 cols) — weekly supply review by DC
- `prod_supply_chain.gold.gsr_sales_history_weekly` (4 cols) — weekly sales actuals

**Join Keys**: `ItemNumber` + `Week` + `Primary_DC`

> **🎯 TARGET COLUMN**: `supply_adequacy` — Categorical flag (Insufficient Supply / Adequate but Tight / Surplus) comparing netted requirement against combined on-hand and in-transit inventory at the DC level.

**Supporting Derived Variables**:
- `supply_gap` — `NettedRequirement - (InvGPDCOnHand + InvGPDCTransit + InvASAPIntlDCOnHand)`
- `forecast_vs_actual_variance` — `Forecast - SalesQty`
- `weeks_of_supply` — `InvGPDCOnHand / Forecast`

In [0]:
%sql
-- SCENARIO 4: Global Projection & Supply Review
-- Joins Jothishwar's new projection tables with supply review and sales history

SELECT 
  proj.ItemNumber,
  proj.Week,
  proj.Primary_DC,
  proj.AirLifeSecure,
  proj.ReplenishmentType,
  proj.PrimaryVendor,
  proj.MakevsBuy,

  -- Projection data
  proj.Forecast                                      AS projected_forecast,
  proj.OpenOrderQty,
  proj.InvGPDCOnHand,
  proj.InvASAPIntlDCOnHand,
  proj.InvGPDCQA,
  proj.InvGPDCTransit,

  -- Supply review
  gsr.DataElement,
  gsr.ValueType,
  gsr.StandardCost,

  -- Sales actuals
  sh.SalesQty                                        AS actual_sales_qty,

  -- Summary-level netted requirement
  summary.NettedRequirement,
  summary.TargetStock,
  summary.Forecast                                   AS summary_forecast,

  -- === DERIVED VARIABLES ===

  -- Net requirement gap
  summary.NettedRequirement - (
    COALESCE(proj.InvGPDCOnHand, 0) + 
    COALESCE(proj.InvGPDCTransit, 0) + 
    COALESCE(proj.InvASAPIntlDCOnHand, 0)
  )                                                  AS supply_gap,

  -- Forecast vs actuals variance
  proj.Forecast - COALESCE(sh.SalesQty, 0)           AS forecast_vs_actual_variance,

  -- Weeks of supply
  CASE 
    WHEN proj.Forecast > 0 
    THEN ROUND(proj.InvGPDCOnHand / proj.Forecast, 1)
    ELSE NULL 
  END                                                AS weeks_of_supply,

  -- Supply adequacy flag
  CASE 
    WHEN summary.NettedRequirement > (COALESCE(proj.InvGPDCOnHand, 0) + COALESCE(proj.InvGPDCTransit, 0))
      THEN 'Insufficient Supply'
    WHEN summary.NettedRequirement > 0 
      THEN 'Adequate but Tight'
    ELSE 'Surplus'
  END                                                AS supply_adequacy

FROM prod_supply_chain.gold.global_projection_tool_detail proj
LEFT JOIN prod_supply_chain.gold.global_projection_tool summary
  ON proj.ItemNumber = summary.ItemNumber
  AND proj.Week = summary.Week
  AND proj.Primary_DC = summary.Primary_DC
LEFT JOIN prod_supply_chain.gold.global_supply_review gsr
  ON proj.ItemNumber = gsr.ItemNumber
  AND proj.Week = gsr.Week
  AND proj.Primary_DC = gsr.Primary_DC
LEFT JOIN prod_supply_chain.gold.gsr_sales_history_weekly sh
  ON proj.ItemNumber = sh.ItemNumber
  AND proj.Week = sh.Week
  AND proj.Primary_DC = sh.Primary_DC_Network
LIMIT 100

## Scenario 5: Order-to-Ship Lead Time + Financial Impact

**Tables**:
- `dev_supply_chain.gold.vw_gptimeinqueue` (9 cols) — order processing timestamps, working days in queue
- `dev_sales.gold.vw_gpcurrentbackorder_enriched` (62 cols) — open backorders with pricing
- `dev_finance.gold.vw_artransactions` (55 cols) — AR transactions and payment status

**Join Keys**:
- `Sales Doc Number` (time in queue) = `SOPNumbe` (backorders)
- `CustNmbr` (backorders) = `Customer_Num` (AR transactions)

> **🎯 TARGET COLUMN**: `delay_severity` — Categorical classification (On Track / Minor Delay / Significant Delay / Critical Delay) based on working days in queue, with associated financial impact.

**Supporting Derived Variables**:
- `total_processing_days` — `WorkingDaysDiff + WorkingDays_DC_to_Shipped`
- `revenue_at_risk_sla` — Extended price for orders exceeding 5-day SLA
- `margin_at_risk` — `XTNDPRCE - EXTDCOST`

In [0]:
%sql
-- SCENARIO 5: Order-to-Ship Lead Time + Financial Impact
-- Joins time-in-queue with backorders and AR transactions

SELECT 
  bo.SOPNumbe                                        AS order_number,
  bo.CustNmbr                                        AS customer_number,
  bo.CustName                                        AS customer_name,
  bo.ItemNmbr                                        AS item_number,
  bo.ItemDesc                                        AS item_description,
  bo.SOPWarehouse                                    AS warehouse,
  bo.BackorderQty,
  bo.XTNDPRCE                                        AS extended_price,
  bo.EXTDCOST                                        AS extended_cost,

  -- Time in queue metrics
  tiq.`Entered Order in DC`,
  tiq.`Ready to ship Entered Date`,
  tiq.`Shipped Entered Date`,
  tiq.WorkingDaysDiff                                AS days_order_to_ready,
  tiq.WorkingDays_DC_to_Shipped                      AS days_dc_to_shipped,
  tiq.`Age Batch`,
  tiq.`DC to Shipped Bucket`,

  -- AR status
  ar.Transaction_Status                              AS ar_status,
  ar.Amount                                          AS ar_amount,

  -- === DERIVED VARIABLES ===

  -- Total processing time
  COALESCE(tiq.WorkingDaysDiff, 0) + 
  COALESCE(tiq.WorkingDays_DC_to_Shipped, 0)         AS total_processing_days,

  -- Revenue at risk for delayed orders
  CASE 
    WHEN tiq.WorkingDaysDiff > 5 THEN bo.XTNDPRCE
    ELSE 0 
  END                                                AS revenue_at_risk_sla,

  -- Margin at risk
  bo.XTNDPRCE - bo.EXTDCOST                          AS margin_at_risk,

  -- Delay severity
  CASE 
    WHEN tiq.WorkingDaysDiff IS NULL THEN 'No Queue Data'
    WHEN tiq.WorkingDaysDiff <= 2 THEN 'On Track'
    WHEN tiq.WorkingDaysDiff <= 5 THEN 'Minor Delay'
    WHEN tiq.WorkingDaysDiff <= 10 THEN 'Significant Delay'
    ELSE 'Critical Delay'
  END                                                AS delay_severity

FROM dev_sales.gold.vw_gpcurrentbackorder_enriched bo
LEFT JOIN dev_supply_chain.gold.vw_gptimeinqueue tiq
  ON bo.SOPNumbe = tiq.`Sales Doc Number`
LEFT JOIN dev_finance.gold.vw_artransactions ar
  ON bo.CustNmbr = ar.Customer_Num
WHERE bo.BackorderQty > 0
ORDER BY total_processing_days DESC NULLS LAST
LIMIT 100

## Scenario 6: Salesforce Pipeline vs. Backorder Overlap

**Tables**:
- `dev_supply_chain.gold.vw_sf_tables_enriched` (38 cols) — Salesforce opportunities with product details
- `dev_master.silver.vw_us_gp_item_master_enriched` (68 cols) — item master with classification
- `dev_inventory.gold.mv_usgp_item_inventory_enriched` (92 cols) — current inventory position

**Join Keys**: `Product Number` (SFDC) ↔ `Item Number` (master/inventory)

> **🎯 TARGET COLUMN**: `pipeline_fulfillment_risk` — Categorical risk flag (High Risk / Medium Risk / Low Risk) for open Salesforce opportunities based on whether the product has active backorders or zero availability.

**Supporting Derived Variables**:
- `critical_item_at_risk` — Boolean: TRUE when a Critical Finished Good has zero availability
- `current_backorder_qty` — Active backorder volume for the item
- `win_probability` — Salesforce stage probability to weight pipeline exposure

In [0]:
%sql
-- SCENARIO 6: Salesforce Pipeline vs Backorder Overlap
-- Identifies sales pipeline opportunities at risk due to inventory/backorder issues

SELECT 
  sf.`Opportunity Name`,
  sf.`Opportunity Owner`,
  sf.Type                                            AS opportunity_type,
  sf.Stage,
  sf.`Stage_Status`,
  sf.`Probability (%)`                               AS win_probability,
  sf.`Product Number`,
  sf.`Product Description`,
  sf.ProductClass                                    AS sf_product_class,
  sf.Segment                                         AS sf_segment,
  sf.`Vizient Flag`,

  -- Item master enrichment
  item.`Item Description`,
  item.`Brand`,
  item.`MktCategory`,
  item.`MktSubCategory`,
  item.`Make/Buy`                                    AS make_or_buy,
  item.`Critical Finished Goods`,

  -- Inventory position for the item
  inv.`On Hand`,
  inv.`Available`,
  inv.`Back ordered`                                 AS current_backorder_qty,
  inv.`On order`,
  inv.`In Transit`,

  -- === DERIVED VARIABLES ===

  -- Pipeline at risk flag
  CASE 
    WHEN inv.`Back ordered` > 0 AND inv.`Available` <= 0 
      THEN 'High Risk - Active Backorder, No Availability'
    WHEN inv.`Back ordered` > 0 
      THEN 'Medium Risk - Active Backorder'
    WHEN inv.`Available` <= 0 
      THEN 'Medium Risk - Zero Availability'
    ELSE 'Low Risk'
  END                                                AS pipeline_fulfillment_risk,

  -- Is this a critical item with pipeline exposure?
  CASE 
    WHEN item.`Critical Finished Goods` = TRUE AND inv.`Available` <= 0 
      THEN TRUE 
    ELSE FALSE 
  END                                                AS critical_item_at_risk

FROM dev_supply_chain.gold.vw_sf_tables_enriched sf
LEFT JOIN dev_master.silver.vw_us_gp_item_master_enriched item
  ON sf.`Product Number` = item.`Item Number`
LEFT JOIN dev_inventory.gold.mv_usgp_item_inventory_enriched inv
  ON sf.`Product Number` = inv.`Item Number`
WHERE sf.Stage_Status NOT IN ('Closed Won', 'Closed Lost')
ORDER BY pipeline_fulfillment_risk, win_probability DESC
LIMIT 100

## Scenario 7: Historical Inventory Trends for Planning

**Tables**:
- `dev_inventory.gold.global_inventory_history` (15 cols) — daily inventory snapshots over time
- `dev_master.gold.date_table` (18 cols) — calendar dimension with week/month/quarter
- `dev_inventory.gold.planning_consolidation` (11 cols) — planning targets and thresholds

**Join Keys**: 
- `Date` (history) → `date` (date_table)
- `Item Number` (history) → `Item Number` (planning)

> **🎯 TARGET COLUMN**: `stock_health_status` — Categorical classification (Chronic Understock / Chronic Overstock / Normal) based on the proportion of days an item's inventory falls below minimum or exceeds maximum thresholds over time.

**Supporting Derived Variables**:
- `days_below_target` — Count of days where `inventory_qty < planning_target`
- `days_below_minimum` — Count of days where `inventory_qty < min_inventory`
- `days_overstocked` — Count of days where `inventory_qty > max_inventory`
- `days_in_month` — Total days for percentage calculation

In [0]:
%sql
-- SCENARIO 7: Historical Inventory Trends for Planning
-- Time-series analysis of inventory vs planning targets

WITH daily_inventory AS (
  SELECT 
    hist.`Item Number`,
    hist.`Item Description`,
    hist.Location,
    hist.`Location (groups)`,
    hist.Source,
    hist.`Product Class Global`,
    hist.`Material Type`,
    hist.`Global FG ABC`,
    hist.Date                                        AS snapshot_date,
    hist.Qty                                         AS inventory_qty,
    hist.`USD $`                                     AS inventory_value_usd,

    -- Calendar enrichment
    dt.year,
    dt.quarter,
    dt.year_month,
    dt.year_week,
    dt.weekday,
    dt.month_number,

    -- Planning targets
    plan.target                                      AS planning_target,
    plan.max_inventory,
    plan.min_inventory,
    plan.on_hand                                     AS planned_on_hand,
    plan.available                                   AS planned_available,
    plan.transit_inventory                            AS planned_transit

  FROM dev_inventory.gold.global_inventory_history hist
  LEFT JOIN dev_master.gold.date_table dt
    ON hist.Date = dt.date
  LEFT JOIN dev_inventory.gold.planning_consolidation plan
    ON hist.`Item Number` = plan.`Item Number`
)

SELECT 
  `Item Number`,
  `Item Description`,
  Location,
  Source,
  `Product Class Global`,
  year_month,

  -- Monthly aggregates
  AVG(inventory_qty)                                 AS avg_monthly_qty,
  MIN(inventory_qty)                                 AS min_monthly_qty,
  MAX(inventory_qty)                                 AS max_monthly_qty,
  AVG(inventory_value_usd)                           AS avg_monthly_value,

  -- Planning comparison
  AVG(planning_target)                               AS planning_target,
  AVG(min_inventory)                                 AS min_inventory_threshold,
  AVG(max_inventory)                                 AS max_inventory_threshold,

  -- === DERIVED VARIABLES ===

  -- Days below target in the month
  SUM(CASE WHEN inventory_qty < planning_target THEN 1 ELSE 0 END) 
                                                     AS days_below_target,

  -- Days below minimum
  SUM(CASE WHEN inventory_qty < min_inventory THEN 1 ELSE 0 END) 
                                                     AS days_below_minimum,

  -- Days overstocked
  SUM(CASE WHEN inventory_qty > max_inventory AND max_inventory > 0 THEN 1 ELSE 0 END) 
                                                     AS days_overstocked,

  COUNT(*)                                           AS days_in_month,

  -- Chronic stock health
  CASE 
    WHEN SUM(CASE WHEN inventory_qty < min_inventory THEN 1 ELSE 0 END) > COUNT(*) * 0.5 
      THEN 'Chronic Understock'
    WHEN SUM(CASE WHEN inventory_qty > max_inventory AND max_inventory > 0 THEN 1 ELSE 0 END) > COUNT(*) * 0.5 
      THEN 'Chronic Overstock'
    ELSE 'Normal'
  END                                                AS stock_health_status

FROM daily_inventory
GROUP BY ALL
ORDER BY `Item Number`, year_month
LIMIT 200

## Next Steps & Additional Exploration Ideas

### Suggested Follow-On Scenarios
1. **ICT (Inter-Company Transfer) Analysis**: Join `dev_inventory.gold.ict_transit` with inventory and sales to track inter-warehouse transfers vs. demand
2. **GPO Contract Compliance**: Join `dev_supply_chain.gold.hpg_default_contract_tiers` with sales data to analyze contract tier utilization
3. **Currency Impact on Inventory Valuation**: Join inventory with `dev_finance.gold.vw_sapapollocurrencyconversionbyday` to calculate exchange rate impact on carrying costs
4. **BOM Explosion for Supply Planning**: Use `dev_inventory.gold.gp_bom_explosion` with backorder data to trace component-level shortages
5. **Dormant Customer + Backorder Cross-Reference**: Join `dev_sales.gold.vw_gpdormantcustomersaudit` with backorder history to identify orders from at-risk customer relationships

### Tips for Your Colleague
- **Start with gold layer** tables — they are the most analysis-ready
- **Use `Item Number` as the universal key** — it bridges all 5 domains
- **Check for `_enriched` variants** — Jothishwar has created enriched versions of many views with additional calculated columns
- **Test join cardinality** before scaling — run `COUNT(DISTINCT key)` on both sides to confirm many-to-one relationships
- **The `Source` column** (GP/SAP) is critical for filtering — many tables contain multi-source data